# week 3 EXERCISE: Generating Synthetic Data

* Write models that can generate datasets
* Use a variety of models and prompts for diverse outputs
* Create a Gradio UI for your product 

## Health Tracking Synthetic Data Generator
* Tracking Type: Blood Pressure, Sleep Cycles, Nutrition, Fitness, Healtcare, Hospital
* LLM models: Anthropic Claude, OpenRouter, OpenAi Groq, Ollama
* Using Gradio
* Adopted python code from _Jai_exercise

In [ ]:
import os
import json, re
import zipfile
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import pandas as pd

In [ ]:
load_dotenv(override=True)


anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
if not anthropic_api_key:
    print("Add ANTHROPIC_API_KEY to .env")
else:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:13]}")


openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if not openrouter_api_key:
    print("Add OPENROUTER_API_KEY to .env")
else:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:9]}")


groq_api_key = os.getenv('GROQ_API_KEY')
if not groq_api_key:
    print("Add GROQ_API_KEY to .env")
else:
    print(f"GROQ API Key exists and begins {groq_api_key[:4]}")


In [ ]:
MODELS = {
    
    "Anthropic: sonnet-4-5-20250929": ("claude-sonnet-4-5-20250929", "https://api.anthropic.com/v1/", anthropic_api_key),
    "OpenAI: gpt-oss-120b": ("openai/gpt-oss-120b", "https://api.groq.com/openai/v1", groq_api_key),
    "OpenRouter: gpt-4o-mini": ("openai/gpt-4o-mini", "https://openrouter.ai/api/v1", openrouter_api_key),
    "OpenRouter: claude-3-haiku": ("anthropic/claude-3-haiku", "https://openrouter.ai/api/v1", openrouter_api_key),
    "Ollama: llama3.2": ("llama3.2", "http://localhost:11434/v1", "ollama")
}

In [ ]:
def get_client(model_key):
   
    model_id, base_url, key = MODELS.get(model_key, list(MODELS.values())[0])

    print(model_id)

    return OpenAI(api_key=key or "ollama", base_url=base_url), model_id

In [ ]:
def build_schema_prompt(application):
    return f"""
You are a database architect and an expert Medical Data Scientist
Your sole task is to generate high-quality, realistic, and clinically coherent health-maintenance tracking datasets.

Strictly adhere to the following rules:
1. DATA STRUCTURE: You must output valid, minified JSON matching the exact schema provided. 
2. MEDICAL COHERENCE: Ensure the fields within 'metrics' perfectly align with the selected 'category' type. Ensure values are physiologically realistic.
3. LOGICAL RELATION: The 'notes' string must logically reflect the underlying data (e.g., if blood pressure is 145/95, notes might mention feeling stressed or drinking coffee).

Create a realistic relational database schema for:

{application}

Requirements:
- 4 to 6 tables
- Include primary keys
- Include foreign keys
- Use realistic column names

Return ONLY valid JSON.

{{
  "application":"",
  "tables":[
    {{
      "table_name":"",
      "description":"",
      "columns":[
        {{
          "name":"",
          "type":"",
          "primary_key":false,
          "foreign_key":""
        }}
      ]
    }}
  ]
}}
"""

In [ ]:
def build_table_prompt(
    table_schema,
    rows
):
    return f"""
Generate {rows} realistic records.

Table Schema:

{json.dumps(table_schema, indent=2)}

Rules:
- Return ONLY JSON array.
- Generate realistic values.
- No markdown.
- No explanation.

Example:

[
  {{
    "id": 1
  }}
]
"""

In [ ]:
def generate_model(
    model_key,
    prompt,
    temperature=0.3
):

    client, model_id = get_client(model_key)  

    response = client.chat.completions.create(
        model=model_id,
        temperature=temperature,
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
def extract_json(text):

    match = re.search(
        r'(\{.*\}|\[.*\])',
        text,
        re.DOTALL
    )

    if not match:
        raise ValueError(
            "JSON not found"
        )

    return json.loads(
        match.group()
    )

In [ ]:
def generate_schema(
    model_key,
    application
):

    prompt = build_schema_prompt(
        application
    )

    response = generate_model(
        model_key,
        prompt
    )

    return extract_json(
        response
    )

In [ ]:
def generate_dataset(
    model_key,
    schema,
    rows
):

    tables = {}

    for table in schema["tables"]:

        prompt = build_table_prompt(
            table,
            rows
        )

        response = generate_model(
            model_key,
            prompt,
            temperature=0.7
        )

        data = extract_json(
            response
        )

        tables[
            table["table_name"]
        ] = pd.DataFrame(data)

    return tables

In [ ]:
def schema_to_markdown(
    schema
):

    md = f"# {schema['application']}\n\n"

    for table in schema["tables"]:

        md += (
            f"## {table['table_name']}\n\n"
        )

        md += "| Column | Type |\n"
        md += "|--------|------|\n"

        for col in table["columns"]:

            md += (
                f"| {col['name']} "
                f"| {col['type']} |\n"
            )

        md += "\n"

    return md

In [ ]:
def preview_table(
    tables
):

    first_table = next(
        iter(tables.values())
    )

    return first_table.head(10)

In [ ]:
def create_zip(
    tables
):

    temp_dir = tempfile.mkdtemp()

    for name, df in tables.items():

        df.to_csv(
            os.path.join(
                temp_dir,
                f"{name}.csv"
            ),
            index=False
        )

    zip_path = os.path.join(
        temp_dir,
        "dataset.zip"
    )

    with zipfile.ZipFile(
        zip_path,
        "w"
    ) as z:

        for name in tables:

            z.write(
                os.path.join(
                    temp_dir,
                    f"{name}.csv"
                ),
                f"{name}.csv"
            )

    return zip_path

In [ ]:
def generate_schema_ui(model_key, application):

    schema = generate_schema(model_key, application)

    markdown = schema_to_markdown(schema)

    return markdown, schema

In [ ]:
def generate_database_ui(
    model_key,
    application,
    rows
):

    # Step 1
    schema = generate_schema(
        model_key,
        application
    )

    # Step 2
    markdown = schema_to_markdown(
        schema
    )

    # Step 3
    tables = generate_dataset(
        model_key,
        schema,
        rows
    )

    # Step 4
    zip_path = create_zip(
        tables
    )

    # Step 5
    first_table = next(
        iter(tables.values())
    )

    status = (
        f"Generated "
        f"{len(tables)} tables "
        f"with {rows} rows each"
    )

    return (
        markdown,
        first_table.head(10),
        zip_path,
        schema,
        tables,
        status
    )

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
        gr.Markdown("# Health Tracking Synthetic Data Generator")
        gr.Markdown("Generate synthetic datasets for Health Management System using your chosen model. ")

        schema_state = gr.State(None)
        dataset_state = gr.State(None)

        with gr.Row():

            model_choice = gr.Dropdown(
                choices=list(MODELS.keys()),
                value="OpenAI: gpt-oss-120b",
                label="Model",
                interactive=True
            )

            application = gr.Dropdown(
                choices=[
                    "Blood Pressure",
                    "Sleep Cycles",
                    "Nutrition",
                    "Fitness",
                    "Healthcare",
                    "Hospital"
                ],
                value="Healthcare",
                label="Tracking",
                allow_custom_value=True
            )

            rows = gr.Slider(
                minimum=10,
                maximum=100,
                value=20,
                step=10,
                label="Rows Per Table"
            )

        with gr.Row():

            generate_database_btn = gr.Button(
                "Generate Database",
                variant="secondary"
            )

        with gr.Tabs():

            with gr.Tab("Download"):

                download_file = gr.File(
                    label="Download Dataset ZIP"
                )


            with gr.Tab("Dataset Preview"):

                dataset_preview = gr.Dataframe(
                    interactive=False
                )

                            
            with gr.Tab("Schema"):

                schema_view = gr.Markdown(
                    value="Schema will appear here..."
                )


        with gr.Accordion(
            "Generation Details",
            open=False
        ):

            generation_status = gr.Textbox(
                label="Status",
                interactive=False
            )

        generate_database_btn.click(
        fn=generate_database_ui,
        inputs=[
            model_choice,
            application,
            rows
        ],
        outputs=[
            schema_view,
            dataset_preview,
            download_file,
            schema_state,
            dataset_state,
            generation_status
        ]
    )

In [ ]:

demo.launch(inbrowser=True)